# Supplementary Tables S1–S7

LaTeX fragments (booktabs `tabular` bodies with their headers; no float, caption or label: those are written in
`Spindle/main_submission.tex`) for the supplementary tables, copied by hand into the manuscript like the figure PDFs.

* **S1** datasets and indexes: sizes, niches, blocks, storage of every index; build time per stage
  (`R1_index_cost`: `results/index_cost/builds.csv`; mean over seeds 0–4, ranges for niches and blocks).
* **S2** hardware and software of the node that ran every build and timing (Slurm, `spindle_env`).
* **S3** complete queries: reading the dataset, the query's logarithms and the searches (`R1_index_cost`:
  `builds.csv`; `R4_timing`: `whole_tile_summary.csv`).
* **S4** partial queries: the query's logarithms and the searches, sizes pooled (`R4_timing`: `gene_set_summary.csv`).
* **S5** cross-dataset partial queries (breast exemplar programmes): Spearman's ρ of closeness to the query and the
  programme's expression over the index tiles, every dataset with maps, seed 0 (`R9_skin_maps`).
* **S6** partial queries from held-out tiles: composition difference to Spindle's first 10 and to a random 10, every
  dataset, |S| = 4 and 16, seed 0 (`R9_skin_maps`).
* **S7** the 24 curated signatures: genes searched, class, Cliff's δ and tiles returned (`R7_queries`: `lists.csv`).

Timing tables: mean ± s.d. over seeds of the per-seed median over the first 200 held-out queries; one thread.
Only existing outputs are used (no new run). Writes `figures/supp_tables/T<k>.tex`: T1–T4 are Tables S1–S4, T7 is S5, T5 is S6 and T6 is S7.

In [1]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

import signatures
import style as ps

OUT = ROOT / "figures" / "supp_tables"
OUT.mkdir(parents=True, exist_ok=True)
R = ROOT / "results"
KEYS = ps.DATASET_ORDER
LAB = ps.DATASET_LABELS


def pm(m, s, f="{:.2f}"):
    return f"{f.format(m)} $\\pm$ {f.format(s)}"


def num(v, f="{:,.0f}"):
    return f.format(v).replace(",", "{,}")


def tabular(cols, header, rows, groups=None):
    """booktabs tabular: cols = column spec, header = list of header cells (may hold \\\\), rows = list of cell lists;
    groups = optional cmidrule header row [(text, first col, last col)]."""
    out = [f"\\begin{{tabular}}{{{cols}}}", "\\toprule"]
    if groups:
        cells, rules, c = [], [], 1
        for text, a, b in groups:
            cells += [""] * (a - c) + [f"\\multicolumn{{{b - a + 1}}}{{c}}{{{text}}}"]
            rules.append(f"\\cmidrule(lr){{{a}-{b}}}")
            c = b + 1
        cells += [""] * (len(header) - c + 1)
        out += [" & ".join(cells) + " \\\\", " ".join(rules)]
    out += [" & ".join(header) + " \\\\", "\\midrule"]
    out += [" & ".join(str(x) for x in r) + " \\\\" for r in rows]
    out += ["\\bottomrule", "\\end{tabular}"]
    return "\n".join(out)


def save(name, *parts):
    text = "\n\n\\medskip\n\n".join(parts) + "\n"
    (OUT / f"{name}.tex").write_text(text)
    print(f"wrote figures/supp_tables/{name}.tex ({len(text.splitlines())} lines)")
    print(text)


def have(*files):
    miss = [str(f.relative_to(ROOT)) for f in files if not f.exists()]
    if miss:
        print("skipped, missing:", miss)
    return not miss

## Table S1 · Datasets and indexes

In [2]:
b = pd.read_csv(R / "index_cost" / "builds.csv")
b["naive_full_s"] = b[["prepare_s", "layout_s", "naive_s"]].sum(axis=1)
g = b.groupby("key")
m = g.mean(numeric_only=True)
rng = lambda c: g[c].agg(lambda s: f"{s.min()}" if s.min() == s.max() else f"{s.min()}--{s.max()}")
niches, blocks = rng("n_niches"), rng("n_blocks")
rows1 = [[LAB[k], num(m.loc[k, "n_cells"]), num(m.loc[k, "p"]), num(m.loc[k, "n_tiles"]), num(m.loc[k, "n_train"]),
          num(m.loc[k, "n_test"]), niches[k], blocks[k],
          *[num(m.loc[k, f"bytes_{c}"] / 1e6, "{:,.1f}") for c in ("whole", "naive", "spindle", "naive_partial",
                                                                   "spindle_partial")]] for k in KEYS]
t1a = tabular("lrrrrrcc" + "r" * 5,
              ["Dataset", "Cells", "Genes", "Tiles", "Index", "Held out", "Niches", "Blocks", "Whole", "Spindle-naive", "Spindle",
               "Spindle-naive (P)", "Spindle-Partial"], rows1,
              groups=[("Tiles", 4, 6), ("Storage (MB)", 9, 13)])
rows2 = [[LAB[k], *[num(m.loc[k, c], "{:,.1f}") for c in ("load_s", "prepare_s", "layout_s", "naive_s", "spindle_s",
                                                          "naive_partial_s", "spindle_partial_s", "build_s")]]
          for k in KEYS]
t1b = tabular("l" + "r" * 8,
              ["Dataset", "Read", "Tiles + covs", "Layout", "Spindle-naive", "Spindle", "Spindle-naive (P)", "Spindle-Partial", "Total"],
              rows2, groups=[("Build time (s)", 2, 9)])
save("T1", t1a, t1b)

wrote figures/supp_tables/T1.tex (35 lines)
\begin{tabular}{lrrrrrccrrrrr}
\toprule
 &  &  & \multicolumn{3}{c}{Tiles} &  &  & \multicolumn{5}{c}{Storage (MB)} \\
\cmidrule(lr){4-6} \cmidrule(lr){9-13}
Dataset & Cells & Genes & Tiles & Index & Held out & Niches & Blocks & Whole & Spindle-naive & Spindle & Spindle-naive (P) & Spindle-Partial \\
\midrule
Skin & 87{,}499 & 382 & 932 & 839 & 93 & 3--4 & 58--82 & 245.5 & 15.9 & 1.8 & 31.8 & 2.2 \\
Kidney & 97{,}560 & 377 & 783 & 705 & 78 & 2--3 & 40--60 & 200.9 & 11.3 & 1.3 & 22.5 & 1.6 \\
Breast & 159{,}226 & 313 & 2{,}044 & 1{,}840 & 204 & 3--4 & 45--67 & 361.7 & 28.5 & 3.2 & 57.0 & 3.9 \\
Lung & 162{,}254 & 377 & 1{,}705 & 1{,}535 & 170 & 3--4 & 57--77 & 437.5 & 28.8 & 3.2 & 57.5 & 3.9 \\
Pancreas & 190{,}965 & 474 & 2{,}290 & 2{,}061 & 229 & 4 & 108--111 & 928.1 & 39.3 & 4.7 & 78.5 & 5.7 \\
Lymph node & 377{,}985 & 377 & 2{,}615 & 2{,}354 & 261 & 5--7 & 103--146 & 670.9 & 41.2 & 4.5 & 82.4 & 5.7 \\
Lymph node 5k & 708{,}983 & 400 & 7{,}

## Table S2 · Hardware and software
The nodes from Slurm (`scontrol show node`; the CPU model is not recorded there), the software from `spindle_env` and the
login node's gcc, which builds the search kernels. The builds and the query timings may run on different nodes
(`host` in builds.csv and the timing CSVs); the cell checks that those nodes have the same hardware.

In [3]:
import platform
import re
import subprocess

import scipy

import data  # noqa: F401  (puts src/ on the path)
from spindle.kernels import _FLAGS

b = pd.read_csv(R / "index_cost" / "builds.csv")
short = lambda hs: sorted({h.split(".")[0] for h in hs})
build_hosts = short(b.host)
timing_hosts = short(set(pd.read_csv(R / "timing" / "whole_tile.csv").host) | set(pd.read_csv(R / "timing" / "gene_set.csv").host)
                     | set(pd.read_csv(R / "parallel" / "timing.csv").host))
HW = ("Sockets", "CoresPerSocket", "ThreadsPerCore", "CPUTot", "RealMemory")
nodes = {}
for h in sorted(set(build_hosts) | set(timing_hosts)):
    out = subprocess.run(["scontrol", "show", "node", h], capture_output=True, text=True, check=True).stdout
    nodes[h] = (dict(re.findall(r"(\w+)=(\S+)", out)), re.search(r"OS=Linux (\S+)", out).group(1).split("-")[0])
assert len({tuple(f[k] for k in HW) + (o,) for f, o in nodes.values()}) == 1, "the nodes differ in hardware or OS"
f, os_rel = next(iter(nodes.values()))
roles = (f"{', '.join(build_hosts)}" if build_hosts == timing_hosts else
         f"{', '.join(build_hosts)} (index builds) and {', '.join(timing_hosts)} (query timings)")
gcc = subprocess.run(["gcc", "--version"], capture_output=True, text=True, check=True).stdout.splitlines()[0]
gcc = re.search(r"\) (\S+)$", gcc).group(1)
cores = int(f["CPUTot"]) // int(f["ThreadsPerCore"])
rows = [["Node" if len(nodes) == 1 else "Nodes",
         f"{roles}: {'each ' if len(nodes) > 1 else ''}{f['Sockets']} sockets, {cores} cores ({f['CPUTot']} hardware "
         f"threads), {int(f['RealMemory']) / 1024:.0f} GB memory"],
        ["Operating system", f"Ubuntu 24.04 (Linux {os_rel}), Slurm; no GPU"],
        ["Software", f"Python {platform.python_version()}, NumPy {np.__version__}, SciPy {scipy.__version__}, "
                     f"pandas {pd.__version__}"],
        ["Search kernels", f"C with AVX2, called through ctypes; gcc {gcc}, " + " ".join(_FLAGS[:5])],
        ["Index builds", f"{int(b.cpus.iloc[0])} CPU cores per build (Slurm)"],
        ["Query timing", "one thread (OpenMP, OpenBLAS and MKL threads set to 1), one timing job at a time"],
        ["Multi-core query timing", "1--64 physical cores, one hardware thread per core, plus one core for the "
                                    "main process; the node was not reserved (other users' jobs could run alongside)"]]
rows = [[a, v.replace("_", "\\_").replace("%", "\\%")] for a, v in rows]
save("T2", tabular("lp{0.72\\textwidth}", ["", ""], rows))

wrote figures/supp_tables/T2.tex (13 lines)
\begin{tabular}{lp{0.72\textwidth}}
\toprule
 &  \\
\midrule
Nodes & ROSALIND-CPU1 (index builds) and ROSALIND-CPU2 (query timings): each 2 sockets, 192 cores (384 hardware threads), 755 GB memory \\
Operating system & Ubuntu 24.04 (Linux 6.8.0), Slurm; no GPU \\
Software & Python 3.12.14, NumPy 2.5.3, SciPy 1.18.1, pandas 3.0.5 \\
Search kernels & C with AVX2, called through ctypes; gcc 13.3.0, -O3 -mavx2 -ffp-contract=off -fno-fast-math -fno-builtin-pow \\
Index builds & 8 CPU cores per build (Slurm) \\
Query timing & one thread (OpenMP, OpenBLAS and MKL threads set to 1), one timing job at a time \\
Multi-core query timing & 1--64 physical cores, one hardware thread per core, plus one core for the main process; the node was not reserved (other users' jobs could run alongside) \\
\bottomrule
\end{tabular}



## Table S3 · Complete queries: reading the data and the stages of a query
Raw read: `load_s` of the builds (reading the h5ad), mean over seeds. Per query (`R4_timing`, mean ± s.d. over seeds of
the per-seed median over the first 200 held-out queries, one thread): the query's logarithms (whole matrix; the blocks
of every niche, shared by Spindle-naive and Spindle), the searches and the end-to-end totals (logarithms +
search summed per query, then the median).

In [4]:
import whole_tile as wt  # end-to-end totals summed per query, as Fig 2B

_, ws = wt.timing_tables(pd.read_csv(R / "timing" / "whole_tile.csv"))
ws = ws.set_index("key")
read = b.groupby("key").load_s.agg(["mean", "std"])
rows = []
for k in KEYS:
    r = ws.loc[k]
    rows.append([LAB[k], pm(read.loc[k, "mean"], read.loc[k, "std"], "{:.1f}"),
                 pm(r.qlog_whole_ms, r.qlog_whole_ms_sd, "{:.1f}"), pm(r.whole_ms, r.whole_ms_sd, "{:.1f}"),
                 f"{r.e2e_whole_ms:.1f}",
                 pm(r.qlog_block_ms, r.qlog_block_ms_sd, "{:.1f}"), pm(r.naive_ms, r.naive_ms_sd),
                 pm(r.spindle_ms, r.spindle_ms_sd), f"{r.e2e_spindle_ms:.1f}"])
save("T3", tabular("l" + "r" * 8, ["Dataset", "Read (s)", "Logarithm", "Scan", "Total", "Block logarithms",
                                   "Spindle-naive", "Spindle", "Spindle total"], rows,
                   groups=[("Whole matrix (ms)", 3, 5), ("Block representation (ms)", 6, 9)]))

wrote figures/supp_tables/T3.tex (16 lines)
\begin{tabular}{lrrrrrrrr}
\toprule
 &  & \multicolumn{3}{c}{Whole matrix (ms)} & \multicolumn{4}{c}{Block representation (ms)} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-9}
Dataset & Read (s) & Logarithm & Scan & Total & Block logarithms & Spindle-naive & Spindle & Spindle total \\
\midrule
Skin & 3.0 $\pm$ 0.3 & 8.8 $\pm$ 0.4 & 16.3 $\pm$ 0.7 & 25.2 & 4.7 $\pm$ 0.6 & 1.09 $\pm$ 0.04 & 0.24 $\pm$ 0.02 & 5.0 \\
Kidney & 3.3 $\pm$ 0.4 & 8.3 $\pm$ 0.1 & 14.4 $\pm$ 0.1 & 22.7 & 2.6 $\pm$ 0.6 & 0.76 $\pm$ 0.07 & 0.16 $\pm$ 0.02 & 2.8 \\
Breast & 4.0 $\pm$ 0.6 & 5.4 $\pm$ 0.1 & 24.1 $\pm$ 0.2 & 29.5 & 3.7 $\pm$ 0.4 & 1.70 $\pm$ 0.09 & 0.39 $\pm$ 0.01 & 4.1 \\
Lung & 3.6 $\pm$ 0.1 & 8.3 $\pm$ 0.1 & 30.1 $\pm$ 0.2 & 38.4 & 4.7 $\pm$ 0.5 & 1.98 $\pm$ 0.07 & 0.49 $\pm$ 0.02 & 5.2 \\
Pancreas & 4.7 $\pm$ 1.8 & 14.2 $\pm$ 0.1 & 73.8 $\pm$ 0.2 & 88.0 & 6.1 $\pm$ 0.0 & 2.84 $\pm$ 0.04 & 0.55 $\pm$ 0.01 & 6.7 \\
Lymph node & 3.5 $\pm$ 0.4 & 8.7 $\pm$ 0.1 & 41.7

## Table S4 · Partial queries: the stages of a query (sizes pooled)
`R4_timing` (`gene_set_summary.csv`), mean ± s.d. over seeds of the per-seed median over the gene-set queries of the
first 200 held-out tiles (|S| = 4, 8, 16, 32), one thread.

In [5]:
import gene_set_accuracy as gsa

_, _, gsu = gsa.timing_tables(pd.read_csv(R / "timing" / "gene_set.csv"))
gsu = gsu.set_index("key")
rows = []
for k in KEYS:
    r = gsu.loc[k]
    rows.append([LAB[k], pm(r.qlog_naive_partial_ms, r.qlog_naive_partial_ms_sd),
                 pm(r.naive_partial_search_ms, r.naive_partial_search_ms_sd, "{:.1f}"),
                 pm(r.qlog_spindle_partial_ms, r.qlog_spindle_partial_ms_sd),
                 pm(r.spindle_partial_ms, r.spindle_partial_ms_sd, "{:.3f}"),
                 f"{r.e2e_spindle_partial_ms:.2f}",
                 pm(r.padding_ms, r.padding_ms_sd, "{:.1f}"), pm(r.imputation_ms, r.imputation_ms_sd, "{:.1f}")])
save("T4", tabular("l" + "r" * 7, ["Dataset", "Block logarithms", "Scan", "Piece logarithms", "Search", "Total",
                                   "Padding", "Imputation"], rows,
                   groups=[("Spindle-naive (ms)", 2, 3), ("Spindle-Partial (ms)", 4, 6), ("Completion (ms)", 7, 8)]))

wrote figures/supp_tables/T4.tex (16 lines)
\begin{tabular}{lrrrrrrr}
\toprule
 & \multicolumn{2}{c}{Spindle-naive (ms)} & \multicolumn{3}{c}{Spindle-Partial (ms)} & \multicolumn{2}{c}{Completion (ms)} \\
\cmidrule(lr){2-3} \cmidrule(lr){4-6} \cmidrule(lr){7-8}
Dataset & Block logarithms & Scan & Piece logarithms & Search & Total & Padding & Imputation \\
\midrule
Skin & 0.52 $\pm$ 0.07 & 4.1 $\pm$ 0.2 & 1.03 $\pm$ 0.14 & 0.098 $\pm$ 0.002 & 1.12 & 4.1 $\pm$ 0.4 & 7.7 $\pm$ 0.7 \\
Kidney & 0.25 $\pm$ 0.06 & 3.4 $\pm$ 0.3 & 0.50 $\pm$ 0.12 & 0.078 $\pm$ 0.009 & 0.58 & 2.5 $\pm$ 0.4 & 4.8 $\pm$ 0.7 \\
Breast & 0.50 $\pm$ 0.07 & 9.0 $\pm$ 0.7 & 1.05 $\pm$ 0.12 & 0.156 $\pm$ 0.003 & 1.21 & 4.1 $\pm$ 0.3 & 7.0 $\pm$ 0.5 \\
Lung & 0.52 $\pm$ 0.08 & 7.6 $\pm$ 0.6 & 1.04 $\pm$ 0.15 & 0.147 $\pm$ 0.005 & 1.17 & 5.1 $\pm$ 0.4 & 8.6 $\pm$ 0.8 \\
Pancreas & 0.63 $\pm$ 0.04 & 6.1 $\pm$ 0.6 & 1.07 $\pm$ 0.04 & 0.154 $\pm$ 0.008 & 1.22 & 7.1 $\pm$ 0.1 & 11.3 $\pm$ 0.1 \\
Lymph node & 0.90 $\pm$ 0.13 

## Table S6 · Partial queries from held-out tiles: composition difference (seed 0)

In [6]:
SM = R / "skin_maps"
GQ = pd.concat([pd.read_csv(SM / "skin_gene_set_queries.csv")]
               + [pd.read_csv(SM / "per_dataset" / k / "seed0_gene_set_queries.csv") for k in KEYS if k != "skin_melanoma"],
               ignore_index=True)
GQ = GQ[(GQ.seed == 0) & GQ["size"].isin([4, 16])]
rows = []
for k in KEYS:
    cells = [LAB[k]]
    for L in (4, 16):
        q = GQ[(GQ.key == k) & (GQ["size"] == L)]
        f = q[q.n_found > 0]
        p = wilcoxon(f.tv_spindle, f.tv_random, alternative="less").pvalue
        cells += [f"{len(f)}/{len(q)}", f"{f.tv_spindle.median():.2f}", f"{f.tv_random.median():.2f}",
                  f"{100 * (f.tv_spindle < f.tv_random).mean():.0f}", f"$<10^{{{int(np.ceil(np.log10(p)))}}}$"]
    rows.append(cells)
save("T5", tabular("l" + "rrrrr" * 2, ["Dataset"] + ["Queries", "Spindle", "Random", "Closer (\\%)", "$p$"] * 2, rows,
                   groups=[("$|S| = 4$", 2, 6), ("$|S| = 16$", 7, 11)]))

wrote figures/supp_tables/T5.tex (16 lines)
\begin{tabular}{lrrrrrrrrrr}
\toprule
 & \multicolumn{5}{c}{$|S| = 4$} & \multicolumn{5}{c}{$|S| = 16$} \\
\cmidrule(lr){2-6} \cmidrule(lr){7-11}
Dataset & Queries & Spindle & Random & Closer (\%) & $p$ & Queries & Spindle & Random & Closer (\%) & $p$ \\
\midrule
Skin & 93/93 & 0.38 & 0.57 & 84 & $<10^{-12}$ & 93/93 & 0.36 & 0.58 & 88 & $<10^{-12}$ \\
Kidney & 78/78 & 0.33 & 0.35 & 67 & $<10^{-3}$ & 78/78 & 0.31 & 0.35 & 72 & $<10^{-4}$ \\
Breast & 204/204 & 0.38 & 0.53 & 69 & $<10^{-9}$ & 204/204 & 0.43 & 0.53 & 60 & $<10^{-5}$ \\
Lung & 170/170 & 0.39 & 0.44 & 74 & $<10^{-11}$ & 170/170 & 0.38 & 0.45 & 67 & $<10^{-9}$ \\
Pancreas & 229/229 & 0.39 & 0.49 & 72 & $<10^{-14}$ & 229/229 & 0.38 & 0.49 & 75 & $<10^{-16}$ \\
Lymph node & 261/261 & 0.37 & 0.41 & 67 & $<10^{-10}$ & 261/261 & 0.33 & 0.41 & 74 & $<10^{-20}$ \\
Lymph node 5k & 774/774 & 0.37 & 0.40 & 63 & $<10^{-19}$ & 774/774 & 0.33 & 0.40 & 78 & $<10^{-65}$ \\
Brain & 1062/1062 & 0.32

## Table S7 · Curated signatures

In [7]:
SIG_CLASS = {"Glioma_Tumor_Proliferative": "Tumor / epithelial", "Basal_Myoepithelial": "Tumor / epithelial",
             "Luminal_Tumor_Core": "Tumor / epithelial", "Proliferation_Signature": "Tumor / epithelial",
             "Tubular_Epithelium": "Tumor / epithelial", "Alveolar_Airway_Epithelium": "Tumor / epithelial",
             "Endocrine_Islet": "Tumor / epithelial", "Exocrine_Ductal": "Tumor / epithelial",
             "Keratinocyte_Epidermis": "Tumor / epithelial", "Melanocyte_Tumor": "Tumor / epithelial",
             "TIL_Lymphocyte": "Lymphoid", "B_Cell_Germinal_Center": "Lymphoid", "T_Cell_Zone": "Lymphoid",
             "Immune_Infiltrate": "Lymphoid", "Injury_Immune_Infiltrate": "Lymphoid",
             "Microglia_Myeloid": "Myeloid", "Macrophage_Myeloid": "Myeloid", "Macrophage_DC_Niche": "Myeloid",
             "Endothelial_Vascular": "Stroma / vascular", "Vascular_Endothelial": "Stroma / vascular",
             "Lymphatic_Vascular_Stroma": "Stroma / vascular", "Mast_Cell_Stroma": "Stroma / vascular"}  # as F4_skin
L = pd.read_csv(R / "queries" / "lists.csv")
L = L[L.method == "spindle_partial"]
print(len(L))
rows = []
for k in KEYS:
    for _, r in L[L.key == k].iterrows():
        genes = signatures.TISSUE_MODULES[signatures.TISSUE[k]][r.list]["genes"]
        miss = set(str(r.missing).split(";")) if isinstance(r.missing, str) else set()
        used = [g for g in genes if g not in miss]
        assert len(used) == int(r.n_genes), (k, r.list, used, r.n_genes)
        rows.append([LAB[k], r.list.replace("_", " "), SIG_CLASS[r.list], ", ".join(used),
                     pm(r.delta, r.delta_sd), f"{r.n_returned:.1f}"])
save("T6", tabular("lll>{\\raggedright\\arraybackslash}p{5.2cm}rr", ["Dataset", "Signature", "Class", "Genes searched", "Cliff's $\\delta$",
                                      "Tiles returned"], rows))

24
wrote figures/supp_tables/T6.tex (30 lines)
\begin{tabular}{lll>{\raggedright\arraybackslash}p{5.2cm}rr}
\toprule
Dataset & Signature & Class & Genes searched & Cliff's $\delta$ & Tiles returned \\
\midrule
Skin & Melanocyte Tumor & Tumor / epithelial & TYR, TYRP1, MLANA, PMEL, DCT, MITF, SOX10, S100B & 0.81 $\pm$ 0.02 & 8.6 \\
Skin & Keratinocyte Epidermis & Tumor / epithelial & KRT1, KRT5, KRT17, LOR & 0.79 $\pm$ 0.02 & 7.0 \\
Skin & Immune Infiltrate & Lymphoid & CD3D, CD3E, CD68, C1QA, ITGAX & 0.88 $\pm$ 0.03 & 3.0 \\
Kidney & Tubular Epithelium & Tumor / epithelial & SLC22A8, SLC4A1, AQP2, UMOD, SLC26A3 & 0.65 $\pm$ 0.05 & 10.0 \\
Kidney & Injury Immune Infiltrate & Lymphoid & HAVCR2, CD68, CD163, CD3D, CD3E & 0.28 $\pm$ 0.12 & 6.4 \\
Breast & Luminal Tumor Core & Tumor / epithelial & ESR1, PGR, ERBB2, FOXA1, GATA3, KRT8, EPCAM, CDH1 & 0.64 $\pm$ 0.02 & 10.0 \\
Breast & Macrophage Myeloid & Myeloid & CD68, CD163, MRC1, C1QA, ITGAX & 0.74 $\pm$ 0.02 & 7.6 \\
Breast & Basal Myoep

## Table S5 · Cross-dataset partial queries: closeness against expression (seed 0)

In [8]:
from scipy.stats import spearmanr

# Spearman's rho of closeness (-d_B^S, exact,
# Spindle-naive) and the programme's tile expression score over the index tiles; breast: the exemplar tile left out
EXEMPLAR = pd.read_csv(R / "pan_cancer" / "exemplar.csv").set_index("program")
PROGS = [("Myeloid", "Myeloid"), ("Cytotoxic_T", "Cytotoxic T cell")]
rows, rho = [], {}
for k in KEYS:
    f = R / "skin_maps" / "per_dataset" / k / "seed0_maps.npz"
    if not f.exists():
        print(f"{k}: no maps (programme genes not all among the index genes)")
        continue
    z = np.load(f, allow_pickle=True)
    cells = [LAB[k]]
    for prog, _ in PROGS:
        keep = np.ones(len(z["tiles"]), bool)
        if k == "breast_cancer":
            keep[z["tiles"] == int(EXEMPLAR.loc[prog, "tile"])] = False
        rho[k, prog] = spearmanr(-z[f"d_{prog}"][keep], z[f"tile_score_{prog}"][keep]).statistic
        cells.append(f"{rho[k, prog]:.2f}")
    cells.append(num(len(z["tiles"])))
    rows.append(cells)
print(pd.Series(rho).unstack().round(2))
save("T7", tabular("lrrr", ["Dataset", "Myeloid", "Cytotoxic T cell", "Index tiles"], rows,
                   groups=[("Spearman's $\\rho$", 2, 3)]))

lymph_node_5k: no maps (programme genes not all among the index genes)
                    Cytotoxic_T  Myeloid
brain_cancer              -0.18     0.35
breast_cancer              0.41     0.53
kidney_nondiseased        -0.08     0.37
lung_cancer                0.29     0.56
lymph_node                 0.75     0.52
pancreatic_cancer         -0.15     0.28
skin_melanoma              0.37     0.45
wrote figures/supp_tables/T7.tex (15 lines)
\begin{tabular}{lrrr}
\toprule
 & \multicolumn{2}{c}{Spearman's $\rho$} &  \\
\cmidrule(lr){2-3}
Dataset & Myeloid & Cytotoxic T cell & Index tiles \\
\midrule
Skin & 0.45 & 0.37 & 839 \\
Kidney & 0.37 & -0.08 & 705 \\
Breast & 0.53 & 0.41 & 1{,}840 \\
Lung & 0.56 & 0.29 & 1{,}535 \\
Pancreas & 0.28 & -0.15 & 2{,}061 \\
Lymph node & 0.52 & 0.75 & 2{,}354 \\
Brain & 0.35 & -0.18 & 9{,}558 \\
\bottomrule
\end{tabular}

